# 02 - Feature Engineering
Notebook wrapper for src/feature_engineering.py.

Builds data/features.parquet and runs quality checks.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Using project root: {PROJECT_ROOT}')

In [ ]:
from src.feature_engineering import build_features

features = build_features(
    data_dir=Path('data'),
    output_path=Path('data/features.parquet'),
    seed=42,
)

print('Rows:', len(features))
print('Columns:', len(features.columns))
features.head()

In [ ]:
null_count = int(features.isna().sum().sum())
default_rate = float(features['default_flag'].mean())

print('Total null values:', null_count)
print('Default rate:', round(default_rate, 4))
features.dtypes.head(20)

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

y = features['default_flag'].astype(int)
candidate_cols = [c for c in features.columns if c not in ['borrower_id', 'default_flag']]

rows = []
for col in candidate_cols:
    x = pd.to_numeric(features[col], errors='coerce').fillna(0.0)
    if x.nunique() <= 1:
        continue
    try:
        auc = roc_auc_score(y, x)
        auc = max(auc, 1.0 - auc)
        rows.append((col, auc))
    except Exception:
        pass

ranked = pd.DataFrame(rows, columns=['feature', 'single_var_auc']).sort_values('single_var_auc', ascending=False)
ranked.head(15)